# 05 — Resume an interrupted run

If an earlier run was killed (machine slept, OOM, kill -9, kernel restart…), re-run the **same command** with `--resume` appended. The trainer reads `outputs/<RUN>/{model_latest.h5, history.json, state.json, replay.npz}`, skips every (round, file) pair already done, and continues. At most one file's worth of work is lost.


In [ ]:
import os, json
ROOT = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.getcwd())=='notebooks' else os.path.abspath('.')
SRC = os.path.join(ROOT, 'src'); DATA = os.path.join(ROOT, 'MMS-FPI-Data-Gaps')

RUN_NAME = 'unet_from_notebook'   # SAME RUN_NAME as 04_train_from_scratch.ipynb
OUT = os.path.join(ROOT, 'outputs', RUN_NAME)

# inspect what's been done so far
h = json.load(open(os.path.join(OUT, 'history.json')))
print(f'{len(h["history"])} files processed so far; last:', h['history'][-1])
print('args were:', h.get('args'))


In [ ]:
# rebuild the SAME command (use the args from the previous run + --resume)
args = h['args']
cmd = (f'python {SRC}/train_incremental.py --data-root {DATA} --out {OUT} '
       f'--model {args["model"]} '
       + (f'--temporal-window {args["temporal_window"]} ' if args.get('temporal_window') else '')
       + (f'--features {args["features"]} ' if args.get('features') else '')
       + (f'--physics-head ' if args.get('physics_head') else '')
       + (f'--dump-embeddings ' if args.get('dump_embeddings') else '')
       + f'--mask {args["mask"]} --rounds {args["rounds"]} '
         f'--inner-epochs {args["inner_epochs"]} --subsample {args["subsample"]} '
         f'--base-filters {args["base_filters"]} --batch-size {args["batch_size"]} '
         f'--val-cap {args["val_cap"]} --replay-per-file {args["replay_per_file"]} '
         f'--replay-cap {args["replay_cap"]} --lr {args["lr"]} '
         f'--resume')
print(cmd)


In [ ]:
# !{cmd}    # uncomment to actually resume
